# Pemetaan & Klasifikasi Lahan Sawah dan Pemukiman Menggunakan Sentinel-2A & openEO

Notebook ini berisi alur kerja lengkap klasifikasi tutupan lahan **Sawah** dan **Pemukiman** di Kecamatan Bungah, Kabupaten Gresik menggunakan citra satelit optik **Sentinel-2A (Level-2A)** via platform **openEO** (Copernicus Data Space Ecosystem) dan pemodelan Machine Learning **Random Forest Classifier**.

--- 
### 📌 Transparansi Pembagian Dataset (Data Training & Testing)
Untuk mencegah kebocoran data (*data leakage*) akibat autokorelasi spasial antar piksel dalam poligon yang sama, pembagian dataset sebesar **70% Training** dan **30% Testing** dilakukan secara terstratifikasi pada tingkat **Poligon Sampel (Group-based Split)**.

Berkas CSV hasil pembagian diekspor secara terpisah ke dalam direktori `data/csv/split_dataset/` agar dapat diperiksa dengan mudah:
1. **Tingkat Poligon (Area Sampel):**
   - `data_training_poligon.csv` (70 Poligon: 35 Sawah, 35 Pemukiman)
   - `data_testing_poligon.csv` (30 Poligon: 15 Sawah, 15 Pemukiman)
2. **Tingkat Piksel (Titik Data Fitur Spektral):**
   - `fitur_piksel_training.csv` (8.618 Piksel Data Latih)
   - `fitur_piksel_testing.csv` (3.419 Piksel Data Uji)

## 1. Data Masukan & Pembagian Dataset Poligon (Train vs Test)

Membaca data geospasial `data/geojson/input2.geojson` (101 geometri: 1 Batas Daerah, 50 Poligon Sawah, dan 50 Poligon Pemukiman), memberikan ID Unik untuk setiap poligon (`Sawah_01..50`, `Pemukiman_01..50`), serta membagi dataset secara transparan menjadi data **Training (70%)** dan **Testing (30%)** pada tingkat poligon sampel.

In [1]:
import openeo, json, os
import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Read GeoJSON Dataset (input2.geojson)
gdf = gpd.read_file("../data/geojson/input2.geojson")
print("=== TOTAL POLIGON MASUKAN GEOJSON ===")
print(gdf["Type"].value_counts())

# Ambil Poligon Sawah dan Pemukiman
samples = gdf[gdf["Type"].isin(["Sawah", "Pemukiman"])].copy().reset_index(drop=True)

# Berikan ID Poligon Unik untuk melacak setiap titik sampel
sawah_cnt, pemukiman_cnt = 1, 1
id_poligon = []
for t in samples["Type"]:
    if t == "Sawah":
        id_poligon.append(f"Sawah_{sawah_cnt:02d}")
        sawah_cnt += 1
    else:
        id_poligon.append(f"Pemukiman_{pemukiman_cnt:02d}")
        pemukiman_cnt += 1
samples["ID_Poligon"] = id_poligon

# Pembagian Stratified Train-Test (70% Training, 30% Testing) berdasarkan ID Poligon
train_ids, test_ids = train_test_split(
    samples["ID_Poligon"],
    test_size=0.30,
    stratify=samples["Type"],
    random_state=42
)
samples["Status_Split"] = np.where(samples["ID_Poligon"].isin(train_ids), "Training", "Testing")

# Ekspor Laporan Pembagian Dataset Poligon ke File CSV di Folder Dedicated
split_dir = "../data/csv/split_dataset"
os.makedirs(split_dir, exist_ok=True)
samples[["ID_Poligon", "Type", "Status_Split"]].to_csv("../data/csv/pembagian_dataset_train_test.csv", index=False)
samples[samples["Status_Split"] == "Training"][["ID_Poligon", "Type"]].to_csv(os.path.join(split_dir, "data_training_poligon.csv"), index=False)
samples[samples["Status_Split"] == "Testing"][["ID_Poligon", "Type"]].to_csv(os.path.join(split_dir, "data_testing_poligon.csv"), index=False)

print("\n=== RINGKASAN PEMBAGIAN DATASET POLIGON (TRAINING vs TESTING) ===")
print(pd.crosstab(samples["Type"], samples["Status_Split"], margins=True))
print(f"\n[SUCCESS]: File CSV Poligon Latih (35 Sawah, 35 Pemukiman) & Uji (15 Sawah, 15 Pemukiman) disimpan di '{split_dir}/'")

# 2. Unduh/Cek Citra Satellite Sentinel-2A via openEO atau File Lokal
tif_target = "../data/tif/sentinel2_area_studi.tif"
if os.path.exists(tif_target):
    print(f"\n[INFO]: File raster lokal '{tif_target}' sudah tersedia.")
else:
    try:
        daerah = gdf[gdf["Type"] == "Daerah"].geometry.iloc[0]
        geom = json.loads(gpd.GeoSeries([daerah]).to_json())["features"][0]["geometry"]
        conn = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
        bands = ["B02", "B03", "B04", "B08", "B11", "SCL"]
        cube = conn.load_collection(
            "SENTINEL2_L2A",
            spatial_extent=geom,
            temporal_extent=["2025-08-24", "2026-08-23"],
            bands=bands,
            max_cloud_cover=30,
        )
        scl = cube.band("SCL")
        mask = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)
        cube = cube.mask(mask.resample_cube_spatial(cube)).filter_bands(bands[:-1])
        cube = cube.reduce_dimension(dimension="t", reducer="median")
        os.makedirs("../data/tif", exist_ok=True)
        cube.download(tif_target, format="GTiff")
        print("\nSUCCESS: Berhasil mengunduh citra Sentinel-2A via openEO.")
    except Exception as e:
        print(f"\n[openEO Info]: {e}")

=== TOTAL POLIGON MASUKAN GEOJSON ===
Type
Sawah        50
Pemukiman    50
Daerah        1
Name: count, dtype: int64

=== RINGKASAN PEMBAGIAN DATASET POLIGON (TRAINING vs TESTING) ===
Status_Split  Testing  Training  All
Type                                
Pemukiman          15        35   50
Sawah              15        35   50
All                30        70  100

[SUCCESS]: File CSV Poligon Latih (35 Sawah, 35 Pemukiman) & Uji (15 Sawah, 15 Pemukiman) disimpan di '../data/csv/split_dataset/'

[INFO]: File raster lokal '../data/tif/sentinel2_area_studi.tif' sudah tersedia.


## 2. Ekstraksi Fitur Spektral & Pembentukan File CSV Piksel Latih/Uji

Mengekstrak nilai reflektansi piksel per poligon sampel (B02, B03, B04, B08, B11), menghitung indeks vegetasi (NDVI), kebasahan (NDWI), dan indeks bangunan (NDBI).

### 📊 Perbedaan Berkas CSV Piksel Latih & Uji:
- **`fitur_piksel_training.csv`**: Berisi seluruh baris piksel (8.618 piksel) yang berasal dari 70 Poligon Training. Data ini yang **dimasukkan ke algoritma Machine Learning untuk melatih model**.
- **`fitur_piksel_testing.csv`**: Berisi seluruh baris piksel (3.419 piksel) yang berasal dari 30 Poligon Testing. Data ini **sama sekali tidak pernah dilihat oleh model saat latihan**, dan digunakan murni untuk **menguji keandalan dan akurasi prediksi model**.

In [1]:
import os, rasterio, numpy as np, pandas as pd
from rasterio.mask import mask as rio_mask

tif_path = "../data/tif/sentinel2_area_studi.tif"
split_dir = "../data/csv/split_dataset"

if os.path.exists(tif_path):
    with rasterio.open(tif_path) as src:
        smp = samples.to_crs(src.crs)
        rows = []
        pix_tr_rows, pix_te_rows = [], []
        
        for idx, r in smp.iterrows():
            try:
                m, _ = rio_mask(src, [r.geometry], crop=True, all_touched=True, filled=False)
            except ValueError:
                continue
            arr = m.astype("float32").filled(np.nan) / 10000
            if src.nodata is not None:
                arr[m.data == src.nodata] = np.nan
            b02, b03, b04, b08, b11 = [arr[k].ravel() for k in range(5)]
            ok = ~np.isnan(b04) & ~np.isnan(b08) & ~np.isnan(b03) & ~np.isnan(b11) & (b04 > 0) & (b08 > 0)
            if ok.sum() == 0:
                continue
            b02_v, b03_v, b04_v, b08_v, b11_v = b02[ok], b03[ok], b04[ok], b08[ok], b11[ok]
            ndvi = (b08_v - b04_v) / (b08_v + b04_v + 1e-6)
            ndwi = (b03_v - b08_v) / (b03_v + b08_v + 1e-6)
            ndbi = (b11_v - b08_v) / (b11_v + b08_v + 1e-6)
            
            rows.append({
                "ID_Poligon": r["ID_Poligon"],
                "Kelas": r["Type"],
                "Status_Split": r["Status_Split"],
                "n_piksel": int(ok.sum()),
                "B02": b02_v.mean(), "B03": b03_v.mean(), "B04": b04_v.mean(),
                "B08": b08_v.mean(), "B11": b11_v.mean(),
                "NDVI": ndvi.mean(), "NDWI": ndwi.mean(), "NDBI": ndbi.mean()
            })
            
            for i in range(len(b02_v)):
                p_row = {
                    "ID_Poligon": r["ID_Poligon"],
                    "Kelas": r["Type"],
                    "B02": round(b02_v[i], 4), "B03": round(b03_v[i], 4), "B04": round(b04_v[i], 4),
                    "B08": round(b08_v[i], 4), "B11": round(b11_v[i], 4),
                    "NDVI": round(ndvi[i], 4), "NDWI": round(ndwi[i], 4), "NDBI": round(ndbi[i], 4)
                }
                if r["Status_Split"] == "Training":
                    pix_tr_rows.append(p_row)
                else:
                    pix_te_rows.append(p_row)
                    
    df_fitur = pd.DataFrame(rows)
    df_pix_tr = pd.DataFrame(pix_tr_rows)
    df_pix_te = pd.DataFrame(pix_te_rows)
    
    print(f"Total Sampel Poligon Valid: {len(df_fitur)}")
    print(f"- Jumlah Piksel Latih (fitur_piksel_training.csv) : {len(df_pix_tr)} baris (dari 70 Poligon Latih)")
    print(f"- Jumlah Piksel Uji   (fitur_piksel_testing.csv)  : {len(df_pix_te)} baris (dari 30 Poligon Uji)")
    print("\n=== RATA-RATA INDEKS SPEKTRAL PER KELAS & SPLIT STATUS ===")
    print(df_fitur.groupby(["Kelas", "Status_Split"])[["n_piksel", "NDVI", "NDWI", "NDBI"]].mean())
    
    os.makedirs("../data/csv", exist_ok=True)
    os.makedirs(split_dir, exist_ok=True)
    df_fitur.to_csv("../data/csv/fitur_sampel.csv", index=False)
    df_pix_tr.to_csv(os.path.join(split_dir, "fitur_piksel_training.csv"), index=False)
    df_pix_te.to_csv(os.path.join(split_dir, "fitur_piksel_testing.csv"), index=False)
    print(f"\n[SUCCESS]: File CSV 'fitur_piksel_training.csv' ({len(df_pix_tr)} baris) & 'fitur_piksel_testing.csv' ({len(df_pix_te)} baris) berhasil diekspor ke '{split_dir}/'")

Total Sampel Poligon Valid: 100
- Jumlah Piksel Latih (fitur_piksel_training.csv) : 8618 baris (dari 70 Poligon Latih)
- Jumlah Piksel Uji   (fitur_piksel_testing.csv)  : 3419 baris (dari 30 Poligon Uji)

=== RATA-RATA INDEKS SPEKTRAL PER KELAS & SPLIT STATUS ===
                          n_piksel      NDVI      NDWI      NDBI
Kelas     Status_Split                                          
Pemukiman Testing       205.933333  0.279766 -0.315628  0.086602
          Training      203.114286  0.294078 -0.333957  0.076638
Sawah     Testing        22.000000  0.368242 -0.276711 -0.234219
          Training       43.114286  0.328709 -0.217390 -0.276136

[SUCCESS]: File CSV 'fitur_piksel_training.csv' (8618 baris) & 'fitur_piksel_testing.csv' (3419 baris) berhasil diekspor ke '../data/csv/split_dataset/'


## 3. Pemodelan Random Forest & Evaluasi Performa (Train vs Test)

Melatih model **Random Forest Classifier** menggunakan piksel dari `fitur_piksel_training.csv` (**8.618 piksel / 70 poligon**) dan mengevaluasi keandalannya secara ketat pada piksel dari `fitur_piksel_testing.csv` (**3.419 piksel / 30 poligon**).

In [1]:
import os, rasterio, numpy as np, pandas as pd
from rasterio.mask import mask as rio_mask
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score, classification_report

TIF = "../data/tif/sentinel2_area_studi.tif"
NAMA_FITUR = ["B02", "B03", "B04", "B08", "B11", "NDVI", "NDWI", "NDBI"]

if os.path.exists(TIF):
    X_tr_list, y_tr_list = [], []
    X_te_list, y_te_list = [], []
    
    with rasterio.open(TIF) as src:
        smp = samples.to_crs(src.crs)
        for idx, r in smp.iterrows():
            try:
                m, _ = rio_mask(src, [r.geometry], crop=True, all_touched=True, filled=False)
            except ValueError:
                continue
            arr = m.astype("float32").filled(np.nan) / 10000
            if src.nodata is not None:
                arr[m.data == src.nodata] = np.nan
            b02, b03, b04, b08, b11 = [arr[k].ravel() for k in range(5)]
            ok = ~np.isnan(b04) & ~np.isnan(b08) & ~np.isnan(b03) & ~np.isnan(b11) & (b04 > 0) & (b08 > 0)
            if ok.sum() == 0:
                continue
            b02_v, b03_v, b04_v, b08_v, b11_v = b02[ok], b03[ok], b04[ok], b08[ok], b11[ok]
            ndvi = (b08_v - b04_v) / (b08_v + b04_v + 1e-6)
            ndwi = (b03_v - b08_v) / (b03_v + b08_v + 1e-6)
            ndbi = (b11_v - b08_v) / (b11_v + b08_v + 1e-6)
            
            feats = np.column_stack([b02_v, b03_v, b04_v, b08_v, b11_v, ndvi, ndwi, ndbi])
            if r["Status_Split"] == "Training":
                X_tr_list.append(feats)
                y_tr_list += [r["Type"]] * len(feats)
            else:
                X_te_list.append(feats)
                y_te_list += [r["Type"]] * len(feats)

    X_train, y_train = np.vstack(X_tr_list), np.array(y_tr_list)
    X_test, y_test = np.vstack(X_te_list), np.array(y_te_list)

    print(f"=== EKSTRAKSI PIKSEL TRAIN vs TEST ===")
    print(f"Jumlah Piksel Training : {len(y_train)} piksel (dari {len(train_ids)} poligon)")
    print(f"Jumlah Piksel Testing  : {len(y_test)} piksel (dari {len(test_ids)} poligon)")
    
    # Melatih Model Random Forest
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    # Prediksi Data Training & Testing
    pred_train = rf.predict(X_train)
    pred_test = rf.predict(X_test)

    labels = ["Sawah", "Pemukiman"]
    print("\n=== EVALUASI PERFORMA MODEL ===")
    print(f"Akurasi Data Training : {accuracy_score(y_train, pred_train) * 100:.2f}%")
    print(f"Akurasi Data Testing  : {accuracy_score(y_test, pred_test) * 100:.2f}%")
    print(f"Cohen's Kappa (Test)  : {cohen_kappa_score(y_test, pred_test):.4f}")
    
    print("\n=== CONFUSION MATRIX (DATA TESTING) ===")
    cm_df = pd.DataFrame(
        confusion_matrix(y_test, pred_test, labels=labels),
        index=[f"Aktual_{l}" for l in labels],
        columns=[f"Prediksi_{l}" for l in labels]
    )
    print(cm_df)

    print("\n=== CLASSIFICATION REPORT (DATA TESTING) ===")
    print(classification_report(y_test, pred_test))

    print("\n=== FEATURE IMPORTANCES ===")
    for n, v in sorted(zip(NAMA_FITUR, rf.feature_importances_), key=lambda t: -t[1]):
        print(f"{n:5s} : {v:.3f}")

    # Menghasilkan & Menyimpan Citra Raster Hasil Klasifikasi
    with rasterio.open(TIF) as src:
        img = src.read().astype("float32") / 10000
        b02, b03, b04, b08, b11 = img[:5]
        ndvi = (b08 - b04) / (b08 + b04 + 1e-6)
        ndwi = (b03 - b08) / (b03 + b08 + 1e-6)
        ndbi = (b11 - b08) / (b11 + b08 + 1e-6)
        
        h, w = b02.shape
        stacked = np.stack([b02, b03, b04, b08, b11, ndvi, ndwi, ndbi], axis=-1).reshape(-1, 8)
        valid_mask = ~np.isnan(stacked).any(axis=1) & (stacked[:, 3] > 0)
        
        pred_map = np.zeros(h * w, dtype=np.uint8)
        if valid_mask.any():
            pred_labels = rf.predict(stacked[valid_mask])
            pred_map[valid_mask] = np.where(pred_labels == "Sawah", 1, 2)
        
        pred_map = pred_map.reshape(h, w)
        meta = src.meta.copy()
        meta.update({"count": 1, "dtype": "uint8", "nodata": 0})
        out_tif = "../data/tif/klasifikasi_sawah_pemukiman.tif"
        with rasterio.open(out_tif, "w", **meta) as dst:
            dst.write(pred_map, 1)
        print("\nSUCCESS: Raster hasil klasifikasi disimpan di '../data/tif/klasifikasi_sawah_pemukiman.tif'")

=== EKSTRAKSI PIKSEL TRAIN vs TEST ===
Jumlah Piksel Training : 8618 piksel (dari 70 poligon)
Jumlah Piksel Testing  : 3419 piksel (dari 30 poligon)

=== EVALUASI PERFORMA MODEL ===
Akurasi Data Training : 99.90%
Akurasi Data Testing  : 99.65%
Cohen's Kappa (Test)  : 0.9797

=== CONFUSION MATRIX (DATA TESTING) ===
                  Prediksi_Sawah  Prediksi_Pemukiman
Aktual_Sawah                 321                   9
Aktual_Pemukiman               3                3086

=== CLASSIFICATION REPORT (DATA TESTING) ===
              precision    recall  f1-score   support

   Pemukiman       1.00      1.00      1.00      3089
       Sawah       0.99      0.97      0.98       330

    accuracy                           1.00      3419
   macro avg       0.99      0.99      0.99      3419
weighted avg       1.00      1.00      1.00      3419


=== FEATURE IMPORTANCES ===
B11   : 0.355
NDBI  : 0.198
B08   : 0.161
B04   : 0.130
B02   : 0.062
NDWI  : 0.054
NDVI  : 0.022
B03   : 0.018

SUCCESS: R

## 4. Visualisasi Peta 3D Interaktif (Plotly Surface)

Menampilkan peta klasifikasi tutupan lahan di atas Digital Elevation Model (DEM) secara interaktif.

In [1]:
import os, rasterio, numpy as np
import plotly.graph_objects as go
from rasterio.warp import reproject, Resampling
from IPython.display import IFrame, display

os.makedirs("../assets/editor/html", exist_ok=True)
html_path = "../assets/editor/html/peta_3d.html"
ref_tif = "../data/tif/klasifikasi_sawah_pemukiman.tif"
dem_tif = "../data/tif/dem_area_studi.tif"

if os.path.exists(ref_tif) and os.path.exists(dem_tif):
    with rasterio.open(ref_tif) as ref, rasterio.open(dem_tif) as dem:
        peta = ref.read(1).astype("float32")
        res = ref.res[0]
        dem_arr = np.zeros(peta.shape, dtype="float32")
        reproject(source=rasterio.band(dem, 1), destination=dem_arr,
                  dst_transform=ref.transform, dst_crs=ref.crs,
                  resampling=Resampling.bilinear)
    peta[peta == 0] = np.nan
    STEP = 3
    Z = dem_arr[::STEP, ::STEP]
    C = peta[::STEP, ::STEP]
    x = np.arange(Z.shape[1]) * res * STEP
    y = np.arange(Z.shape[0]) * res * STEP
    colorscale = [[0, "#2e8b57"], [0.5, "#2e8b57"], [0.5, "#d2691e"], [1, "#d2691e"]]
    fig = go.Figure(go.Surface(
        x=x, y=y, z=Z, surfacecolor=C,
        colorscale=colorscale, cmin=1, cmax=2,
        colorbar=dict(title="Kelas", tickvals=[1.25, 1.75], ticktext=["Sawah", "Pemukiman"]),
        lighting=dict(ambient=0.6, diffuse=0.8, roughness=0.9),
    ))
    fig.update_layout(
        title="Klasifikasi Sawah & Pemukiman (3D Surface Relief)",
        scene=dict(aspectmode="manual", aspectratio=dict(x=1, y=1, z=0.25),
                   xaxis_title="Timur (m)", yaxis_title="Utara (m)", zaxis_title="Elevasi (m)"),
        height=750,
    )
    fig.write_html(html_path)
    print(f"SUCCESS: Peta 3D berhasil disimpan di '{html_path}'")
    display(IFrame(html_path, width="100%", height=750))

SUCCESS: Peta 3D berhasil disimpan di '../assets/editor/html/peta_3d.html'
